### Imoprts and Env
---

In [ ]:
import asyncio
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool, BaseModel
from openai.types.responses import ResponseTextDeltaEvent

In [ ]:
load_dotenv(override=True)

### Creating Agents
---

In [ ]:
instructions1 = "You are a sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write professional, serious cold emails."

instructions2 = "You are a humorous, engaging sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write witty, engaging cold emails that are likely to get a response."

instructions3 = "You are a busy sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write concise, to the point cold emails."

In [ ]:
internAgent = Agent(
    name="Intern Sales Agent",
    instructions=instructions1,
    model="gpt-4o-mini",
)

engagingAgent = Agent(
        name="Engaging Sales Agent",
        instructions=instructions2,
        model="gpt-4o-mini"
)

busyAgent = Agent(
        name="Busy Sales Agent",
        instructions=instructions3,
        model="gpt-4o-mini"
)

In [ ]:
sales_picker = Agent(
    name="Sales Email Picker",
    instructions="You pick the best cold sales email from the given options. \
Imagine you are a customer and pick the one you are most likely to respond to. \
Do not give an explanation; reply with the selected email only.",
    model="gpt-4o-mini"
)

### Running Async Reponse Stream
---

In [ ]:
message = "Write a cold sales email"

In [ ]:
async def main() -> None:
    stream = Runner.run_streamed(
        internAgent, 
        message
    )

    async for event in stream.stream_events():
        if (
            event.type == "raw_response_event"
            and isinstance(event.data, ResponseTextDeltaEvent)
        ):
            print(event.data.delta, end="", flush=True)

    print("Finished running")



### Parallel/Concurrent Calls
---

In [ ]:
with trace("Parallel cold emails"):
    salesOptionsResults = await asyncio.gather(
        Runner.run(internAgent, message),
        Runner.run(engagingAgent, message),
        Runner.run(busyAgent, message),
    )

outputs = [option.final_output for option in salesOptionsResults]

for output in outputs:
    print(output + "\n\n")

In [ ]:
with trace("Selection of email message for sales people"):
    results = await asyncio.gather(
        Runner.run(internAgent, message),
        Runner.run(engagingAgent, message),
        Runner.run(busyAgent, message),
    )

    outputs = [result.final_output for result in results]

    emails = "Cold sales emails:\n\n" + "\n\nEmail:\n\n".join(outputs)

    bestTemplate = await Runner.run(sales_picker, emails)

    print(f"Best sales email:\n{bestTemplate.final_output}")

### Subject & Body Tools
---

In [ ]:
subject_instructions = """
You can write a subject for a cold sales email.
You are given a message and you need to write a subject for an email that is  likely to get a response.
"""

In [ ]:
subject_writer = Agent(
    name="Email Subject Writer",
    instructions=subject_instructions,
    model="gpt-4o-mini",
)

# subject_writer.as_tool("subject_writer", tool_description="Write a subject for a cold sales email")

In [ ]:
html_instructions = """
You can convert a text email body to an HTML email body.
You are given a text email body which might have some markdown and you
need to conver it to an HTML email body with simple, clear, compelling layout and design.
"""

In [ ]:
html_converter = Agent(
    name="HTML Converter",
    instructions=html_instructions,
    model="gpt-4o-mini"
)

# html_converter.as_tool("html_coverter", tool_description="Convert a text email body to an HTML email body")

In [ ]:
@function_tool
def send_html_email(subject: str, html_body: str) -> None:
    """ Send out an email with the given subject and HTML body to all sales prospects """
    print("Email Sent!")
    print(subject, html_body)

### Handoff
---

In [ ]:
emailer_agent_tools = [
    subject_writer.as_tool("subject_writer", tool_description="Write a subject for a cold sales email"),
    html_converter.as_tool("html_coverter", tool_description="Convert a text email body to an HTML email body"),
    send_html_email
]

In [ ]:
manager_instructions = """
You are an email formatter and sender. You receive the body of an email to be sent.
You first use use the subject_writer tool to write a subject for the email, then use the html_converter tool to convert the body to HTML.
Finally, you use the send_html_email tool to  send the email with the subject and HTML body.
"""

In [ ]:
emailer_agent = Agent(
    name="Email Manager",
    instructions=manager_instructions,
    tools=emailer_agent_tools,
    model="gpt-4o-mini",
    handoff_description="Convert an email to HTML and send it"
)

In [ ]:
tool_message = "Write a cold sales email"

sales_manager_tools = [
    internAgent.as_tool("sales_agent1", tool_description=tool_message),
    engagingAgent.as_tool("sales_agent2", tool_description=tool_message),
    busyAgent.as_tool("sales_agent3", tool_description=tool_message),
]

In [ ]:
sales_manager_instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.

Follow these steps carefully:
1. Generate Drafts: Use all thee sales_agent tools to generate three different email drafts. Do not proceed until three drafts are ready.

2. Evaluate and Select: Review the drafts and choose the single best email using your judgement of which one is most effective.
You can use the tools multiple times if you're not satified with the results from the first try.

3. Handoff for Sending: Pass ONLY the winning email draft to the "Email Manager" agent. The Email Manager will take care of fromatting and sending.

Crucial Rules:
- You must use the sales agent tools to generate the drafts - do not write them yourself.
- You must hand off exactly ONE email to the Email Manager - never more than one.
"""

In [ ]:
sales_manager = Agent(
    name="Sales Manager",
    instructions=sales_manager_instructions,
    tools=sales_manager_tools,
    handoffs=[emailer_agent],
    model="gpt-4o-mini",
)

In [ ]:
message = "Send ouyt a cold sales email addessed to Dear CEO from Alice"

with trace("Autmoted SDR"):
    result = await Runner.run(sales_manager, message)

### Guardrails
---

In [ ]:
from openai import BaseModel


class NameCheckOutput(BaseModel):
    